# 03 - Entrenamiento, Evaluación, Calibración de Umbral y Explicabilidad
## Sistema Inteligente de Clasificación Micológica (Secondary Mushroom Dataset)

Este cuaderno implementa el flujo técnico completo del **Hito 2**:
1. **Entrenamiento de 4 Modelos de Machine Learning**: Regresión Logística, Random Forest, XGBoost y Red Neuronal Perceptrón Multicapa (MLP).
2. **Ajuste Asimétrico de Umbral (Cost-Sensitive Learning)**: Metodología de riesgo cero orientada a garantizar $FN=0$ (100% Recall en clase venenosa).
3. **Explicabilidad Global y Local (SHAP y LIME)**: Interpretación de patrones morfológicos determinantes y explicaciones a nivel de instancia.
4. **Serialización y Verificación (`joblib`)**: Exportación de artefactos para consumo en la aplicación interactiva (`app/`).

In [ ]:
import sys
import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Métricas y modelos
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Explicabilidad
import shap
import lime
import lime.lime_tabular

# Módulos del proyecto
sys.path.append(str(Path.cwd().parent))
from src.data_loader import (
    load_raw_data,
    get_train_val_test_data,
    NUMERIC_COLUMNS,
    TARGET_COLUMN
)
from src.models import (
    build_preprocessing_pipeline,
    create_model_pipeline,
    get_candidate_models
)
from src.utils import (
    plot_confusion_matrix,
    evaluate_model_performance,
    evaluate_thresholds,
    find_zero_fn_threshold,
    plot_threshold_tuning_curve
)

print("✓ Entorno y dependencias cargados exitosamente.")

### 1. Carga de Datos y Partición Estratificada (Train / Val / Test)
Separamos el conjunto en **70% Entrenamiento**, **15% Validación** y **15% Prueba**.
Estandarizamos la variable objetivo de forma binaria:
- $Y = 0$: Comestible (`'e'` - Edible)
- $Y = 1$: Venenoso (`'p'` - Poisonous / Positiva para riesgo toxicológico)

In [ ]:
# Carga de datos crudos del Secondary Mushroom Dataset
df = load_raw_data()

# Partición 70 / 15 / 15 estratificada
X_train, X_val, X_test, y_train, y_val, y_test = get_train_val_test_data(df)

# Mapeo binario formal: 0 = 'e' (comestible), 1 = 'p' (venenoso)
TARGET_MAP = {'e': 0, 'p': 1}
INV_TARGET_MAP = {0: 'e', 1: 'p'}

y_train_bin = y_train.map(TARGET_MAP).astype(int)
y_val_bin = y_val.map(TARGET_MAP).astype(int)
y_test_bin = y_test.map(TARGET_MAP).astype(int)

categorical_cols = [c for c in X_train.columns if c not in NUMERIC_COLUMNS]

print(f"Dimensiones Train: {X_train.shape} | Venenosos: {y_train_bin.sum()} ({y_train_bin.mean():.1%})")
print(f"Dimensiones Val:   {X_val.shape} | Venenosos: {y_val_bin.sum()} ({y_val_bin.mean():.1%})")
print(f"Dimensiones Test:  {X_test.shape} | Venenosos: {y_test_bin.sum()} ({y_test_bin.mean():.1%})")

--- 
## 1. Entrenamiento de los 4 Modelos de Machine Learning Requeridos
Evaluamos en la partición de Validación (`X_val`, `y_val_bin`) los cuatro modelos de la propuesta técnica:
1. **Regresión Logística**: Baseline lineal con probabilidades sigmoidales.
2. **Random Forest**: Ensamble no paramétrico de árboles de decisión (Bagging).
3. **XGBoost**: Ensamble de gradiente descendente optimizado (Boosting).
4. **MLPClassifier**: Red Neuronal Artificial Perceptrón Multicapa con regularización y early stopping.

In [ ]:
candidate_models = get_candidate_models()
fitted_pipelines = {}
metrics_list = []

print("=" * 75)
print("ENTRENAMIENTO Y EVALUACIÓN EN CONJUNTO DE VALIDACIÓN")
print("=" * 75)

for model_name, estimator in candidate_models.items():
    print(f"\n>>> Entrenando: {model_name}...")
    pipeline = create_model_pipeline(estimator, NUMERIC_COLUMNS, categorical_cols)
    
    t_start = time.time()
    pipeline.fit(X_train, y_train_bin)
    train_duration = time.time() - t_start
    
    # Predicciones por defecto (umbral 0.5)
    y_val_pred = pipeline.predict(X_val)
    fitted_pipelines[model_name] = pipeline
    
    # Métricas clave orientadas a la toxicidad
    perf = evaluate_model_performance(y_val_bin, y_val_pred, model_name=model_name)
    perf["Train Time (s)"] = round(train_duration, 2)
    metrics_list.append(perf)
    
    print(f"    Entrenado en {train_duration:.2f}s | Acc: {perf['Accuracy']:.4f} | Recall: {perf['Recall (Poisonous)']:.4f} | FN: {perf['FN (Falsos Negativos)']}")

# Tabla comparativa de resultados
df_comparison = pd.DataFrame(metrics_list)
display_cols = ["Model", "Accuracy", "Precision", "Recall (Poisonous)", "F1-Score", "FN (Falsos Negativos)", "FP (Falsos Positivos)", "Train Time (s)"]
df_comparison = df_comparison[display_cols]

print("\n" + "=" * 75)
print("RESUMEN COMPARATIVO DE MODELOS (VALIDACIÓN)")
print("=" * 75)
print(df_comparison.to_string(index=False))

# Visualización comparativa de métricas clave
fig, ax = plt.subplots(figsize=(10, 5))
metrics_to_plot = df_comparison.melt(id_vars="Model", value_vars=["Accuracy", "Precision", "Recall (Poisonous)", "F1-Score"], var_name="Métrica", value_name="Valor")
sns.barplot(data=metrics_to_plot, x="Model", y="Valor", hue="Métrica", palette="Set2", ax=ax)
ax.set_ylim(0.75, 1.02)
ax.set_title("Comparación de Rendimiento de los 4 Modelos (Validación)", fontsize=13, fontweight="bold")
ax.set_ylabel("Puntaje (0 - 1)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

--- 
## 2. Ajuste Asimétrico de Umbral (Cost-Sensitive Learning / Riesgo Cero)

En clasificación toxicológica existe una asimetría crítica en los costos de error:
- **Falso Positivo (FP)**: Un hongo comestible es clasificado como venenoso. Costo: se descarta alimento seguro (pérdida económica o alimentaria moderada).
- **Falso Negativo (FN)**: Un hongo venenoso es clasificado como comestible. Costo: intoxicación severa o fatalidad del consumidor (**Costo Inaceptable**).

### Metodología de Riesgo Cero:
Calculamos $P(Y=1 \mid X)$ con `predict_proba()` y barremos el umbral de decisión $\tau$ desde $0.01$ hasta $0.50$.
Buscamos el umbral óptimo $\tau^*$ tal que:
$$\tau^* = \max \{ \tau \mid FN(\tau) = 0 \}$$

In [ ]:
# Seleccionamos el modelo a calibrar (ej. Logistic Regression muestra claramente el efecto del umbral, o XGBoost/Random Forest)
model_to_tune = "Logistic Regression"
pipeline_to_tune = fitted_pipelines[model_to_tune]

# 1. Extracción de probabilidades en la clase venenosa (Y=1)
y_val_probs = pipeline_to_tune.predict_proba(X_val)[:, 1]

# 2. Barrido exhaustivo de umbrales con nuestra función modular
threshold_grid = np.linspace(0.01, 0.50, 50)
df_thresholds = evaluate_thresholds(y_val_bin.values, y_val_probs, thresholds=threshold_grid)

# 3. Encontrar umbral óptimo de riesgo cero (FN = 0)
optimal_th, best_th_stats = find_zero_fn_threshold(df_thresholds)

print(f"=== Calibración de Umbral para {model_to_tune} ===")
print(f"Umbral por defecto (0.50): FN = {int(df_thresholds.loc[df_thresholds['threshold'] == 0.50, 'FN'].values[0])} hongos venenosos clasificados como comestibles.")
print(f"Umbral Calibrado (Riesgo Cero, tau* = {optimal_th:.4f}): FN = {int(best_th_stats['FN'])}, Recall Venenoso = {best_th_stats['Recall (Poisonous)']:.2%}, Precisión = {best_th_stats['Precision']:.2%}")

# 4. Graficar curva de trade-off de umbral
plot_threshold_tuning_curve(
    df_thresholds, 
    optimal_threshold=optimal_th,
    title=f"Ajuste Asimétrico de Umbral - {model_to_tune} (Objetivo FN=0)"
)

# 5. Comparativa de Matrices de Confusión (Antes vs Después)
y_val_pred_default = (y_val_probs >= 0.50).astype(int)
y_val_pred_calibrated = (y_val_probs >= optimal_th).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_default = confusion_matrix(y_val_bin, y_val_pred_default)
sns.heatmap(cm_default, annot=True, fmt="d", cmap="Blues", xticklabels=["Comestible (0)", "Venenoso (1)"], yticklabels=["Comestible (0)", "Venenoso (1)"], ax=axes[0])
axes[0].set_title(f"Umbral Estándar (tau = 0.50)\nFN = {cm_default[1, 0]} (Peligro)", fontsize=11, fontweight="bold", color="darkred")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Real")

cm_calibrated = confusion_matrix(y_val_bin, y_val_pred_calibrated)
sns.heatmap(cm_calibrated, annot=True, fmt="d", cmap="Greens", xticklabels=["Comestible (0)", "Venenoso (1)"], yticklabels=["Comestible (0)", "Venenoso (1)"], ax=axes[1])
axes[1].set_title(f"Umbral Calibrado (tau* = {optimal_th:.3f})\nFN = {cm_calibrated[1, 0]} (Riesgo Cero Garantizado)", fontsize=11, fontweight="bold", color="darkgreen")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Real")

plt.tight_layout()
plt.show()

--- 
## 3. Explicabilidad Global con SHAP y Local con LIME

Utilizamos técnicas de Inteligencia Artificial Explicable (XAI) sobre nuestro mejor ensamble (`Random Forest`):
- **SHAP (SHapley Additive exPlanations)**: Asigna valores de atribución basados en teoría de juegos para cuantificar la contribución positiva o negativa de cada característica morfológica a nivel global.
- **LIME (Local Interpretable Model-agnostic Explanations)**: Entrena un modelo subrogado linealmente interpretable alrededor de una muestra específica para explicar por qué el sistema tomó dicha decisión.

In [ ]:
# Selección del modelo estelar para explicabilidad: Random Forest
best_model_name = "Random Forest"
best_pipeline = fitted_pipelines[best_model_name]

preprocessor = best_pipeline.named_steps["preprocessor"]
rf_classifier = best_pipeline.named_steps["classifier"]

# Transformación del conjunto de prueba y obtención de nombres de características decodificadas
X_test_proc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print(f"Dimensiones de entrada al clasificador: {X_test_proc.shape}")
print(f"Total características codificadas (One-Hot + Escala): {len(feature_names)}")

# ---------------------------------------------------------
# 3.1 SHAP Global Explainability
# ---------------------------------------------------------
print("\n>>> Calculando valores SHAP con TreeExplainer...")
explainer_shap = shap.TreeExplainer(rf_classifier)

# Muestra representativa para cálculo ágil en notebook
sample_shap_size = 300
X_shap_sample = X_test_proc[:sample_shap_size]
shap_values = explainer_shap.shap_values(X_shap_sample)

# Extracción de contribuciones a la clase venenosa (Y=1)
if isinstance(shap_values, list):
    shap_poisonous = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_poisonous = shap_values[:, :, 1]
else:
    shap_poisonous = shap_values

plt.figure(figsize=(10, 6))
plt.title("SHAP Summary Plot - Importancia Morfológica Global en Toxicidad (Clase Venenosa)", fontsize=12, fontweight="bold")
shap.summary_plot(shap_poisonous, X_shap_sample, feature_names=feature_names, max_display=12, show=False)
plt.tight_layout()
plt.show()

# Gráfico de barras de impacto absoluto medio global
plt.figure(figsize=(10, 5))
plt.title("SHAP Feature Importance (Magnitud Media de Impacto Global)", fontsize=12, fontweight="bold")
shap.summary_plot(shap_poisonous, X_shap_sample, feature_names=feature_names, plot_type="bar", max_display=10, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# 3.2 LIME Local Explainability (Instancia Individual)
# ---------------------------------------------------------
print(">>> Instanciando LimeTabularExplainer...")

# Datos de fondo representativos para muestreo de perturbaciones en LIME
background_data = preprocessor.transform(X_train.iloc[:1000])

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=background_data,
    feature_names=feature_names,
    class_names=["Comestible", "Venenoso"],
    mode="classification",
    random_state=42
)

# Seleccionar una muestra específica de prueba
sample_index = 0
instance_to_explain = X_test_proc[sample_index]
real_label = "Venenoso" if y_test_bin.iloc[sample_index] == 1 else "Comestible"

# Función de predicción probabilística
predict_fn = lambda x: rf_classifier.predict_proba(x)

exp = lime_explainer.explain_instance(
    data_row=instance_to_explain,
    predict_fn=predict_fn,
    num_features=10,
    labels=(1,)
)

print(f"\n=== Explicación LIME para Instancia de Prueba #{sample_index} ===")
print(f"Etiqueta Real: {real_label}")
probs = predict_fn(instance_to_explain.reshape(1, -1))[0]
print(f"Probabilidades del Modelo: P(Comestible)={probs[0]:.3f}, P(Venenoso)={probs[1]:.3f}")

print("\nPonderación local de características (Impacto en clase Venenosa):")
for feature, weight in exp.as_list(label=1):
    impact = "Aumenta Riesgo" if weight > 0 else "Reduce Riesgo (Hacia Comestible)"
    print(f"  • {feature:<40}: {weight:+.4f} ({impact})")

# Visualización gráfica de LIME
fig = exp.as_pyplot_figure(label=1)
plt.title(f"Explicación Local LIME (Instancia #{sample_index} - Real: {real_label})", fontsize=11, fontweight="bold")
plt.xlabel("Peso de Contribución")
plt.tight_layout()
plt.show()

--- 
## 4. Exportación y Serialización de Artefactos con `joblib`

Exportamos los objetos entrenados a la carpeta `app/models/` para ser consumidos de manera eficiente por la aplicación interactiva de Streamlit (`app/app.py`):
1. `preprocessor.joblib`: Transformador de variables numéricas y categóricas (`ColumnTransformer`).
2. `best_model.joblib`: Estimador final optimizado (`RandomForestClassifier`).
3. `pipeline.joblib`: Pipeline unificado de inferencia de punta a punta.
4. `model_metadata.json`: Metadatos técnicos que incluyen el umbral calibrado ($\tau^*$), métricas y esquema de características.

In [ ]:
# Directorio destino en la arquitectura del proyecto
models_dir = Path.cwd().parent / "app" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

preprocessor_path = models_dir / "preprocessor.joblib"
best_model_path = models_dir / "best_model.joblib"
pipeline_path = models_dir / "pipeline.joblib"
metadata_path = models_dir / "model_metadata.json"

# Serialización con compresión joblib
joblib.dump(preprocessor, preprocessor_path, compress=3)
joblib.dump(rf_classifier, best_model_path, compress=3)
joblib.dump(best_pipeline, pipeline_path, compress=3)

# Guardar metadatos técnicos y umbral calibrado
metadata = {
    "model_name": best_model_name,
    "optimal_threshold": float(optimal_th),
    "classes": {"0": "edible", "1": "poisonous"},
    "numeric_features": NUMERIC_COLUMNS,
    "categorical_features": categorical_cols,
    "num_features_encoded": int(len(feature_names))
}
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("✓ Artefactos exportados exitosamente:")
print(f"  • Preprocessor : {preprocessor_path.resolve()} ({preprocessor_path.stat().st_size / 1024:.1f} KB)")
print(f"  • Best Model   : {best_model_path.resolve()} ({best_model_path.stat().st_size / 1024:.1f} KB)")
print(f"  • Full Pipeline: {pipeline_path.resolve()} ({pipeline_path.stat().st_size / 1024:.1f} KB)")
print(f"  • Metadata     : {metadata_path.resolve()}")

### 4.1 Verificación de Carga e Inferencia en Producción
Simulamos el proceso exacto que realizará `app/app.py` al recibir una nueva observación de hongo para clasificar.

In [ ]:
# 1. Cargar artefactos serializados
loaded_preprocessor = joblib.load(preprocessor_path)
loaded_model = joblib.load(best_model_path)
with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)

calibrated_threshold = loaded_meta["optimal_threshold"]

# 2. Tomar un espécimen de prueba no visto
test_sample = X_test.iloc[[5]].copy()
real_status = "Venenoso (p)" if y_test_bin.iloc[5] == 1 else "Comestible (e)"

# 3. Preprocesamiento e inferencia
sample_encoded = loaded_preprocessor.transform(test_sample)
prob_poisonous = loaded_model.predict_proba(sample_encoded)[0, 1]

# Aplicación del umbral de riesgo cero
is_toxic = prob_poisonous >= calibrated_threshold
predicted_label = "Venenoso (p) ⚠️" if is_toxic else "Comestible (e) 🍽️"

print("=== TEST DE INFERENCIA EN PRODUCCIÓN ===")
print(f"Muestra de entrada: {test_sample.to_dict(orient='records')[0]}")
print(f"\nProbabilidad de Toxicidad : {prob_poisonous * 100:.2f}%")
print(f"Umbral de Riesgo Cero      : {calibrated_threshold * 100:.2f}%")
print(f"Dictamen del Sistema       : {predicted_label}")
print(f"Estado Real en Dataset     : {real_status}")
print("\n✓ Verificación exitosa: pipeline listo para despliegue en Streamlit.")